# Prédiction de Toxicité avec Keras et RDKit
Dans ce notebook, nous allons construire un réseau de neurones avec Keras (TensorFlow) pour classifier des molécules (Toxique vs Non-Toxique) en utilisant les empreintes digitales de Morgan (Morgan Fingerprints).

<img src="images/fingerprint.png" alt="Description de l'image" width="400"/>


In [21]:
# Installation des dépendances (décommentez et exécutez si nécessaire)
# !pip install rdkit tensorflow scikit-learn numpy

In [22]:
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras import layers

# 1. DONNÉES D'EXEMPLE
# 1 = Toxique, 0 = Non toxique
data = [
    ("c1ccccc1", 1),       # Benzène
    ("CCO", 0),            # Éthanol
    ("C(=O)O", 0),         # Acide formique
    ("C1=CC=C(C=C1)O", 1), # Phénol
    ("C", 0),              # Méthane
    ("C1=CC=C(C=C1)Cl", 1) # Chlorobenzène
]

smiles_list = [item[0] for item in data]
labels = [item[1] for item in data]

## conversion des molècules en nombres
On part d’une liste de molécules écrites en SMILES.
Avec RDKit, chaque molécule est transformée en un vecteur binaire (fingerprint) qui encode sa structure chimique.
On regroupe tous ces vecteurs dans une matrice X, et les étiquettes (toxique / non toxique) dans y.
On divise ensuite les données en deux parties :

une pour entraîner le modèle
une pour tester ses performances

En résumé : on convertit des molécules en nombres, puis on prépare les données pour entraîner un modèle de machine learning.

In [23]:
# On importe le nouveau générateur moderne
from rdkit.Chem import rdFingerprintGenerator

# 2. CONVERSION EN VECTEURS (Morgan Fingerprints - Méthode Moderne)
def smiles_to_fingerprint(smiles, radius=2, n_bits=2048):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return np.zeros((n_bits,))
    
    # Création du générateur (nouvelle méthode recommandée)
    mfpgen = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=n_bits)
    
    # Génération de l'empreinte
    fp = mfpgen.GetFingerprint(mol)
    
    return np.array(fp)

X = np.array([smiles_to_fingerprint(s) for s in smiles_list])
y = np.array(labels)

# 3. SÉPARATION ENTRAÎNEMENT / TEST
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

print(f"Taille des données d'entraînement : {X_train.shape}")
print(f"Taille des données de test : {X_test.shape}")


Taille des données d'entraînement : (4, 2048)
Taille des données de test : (2, 2048)


## Contruction du réseau de neurones avec Keras

entrée = vecteur fingerprint (2048 valeurs)
couches cachées = apprennent des motifs (64 puis 32 neurones)
Dropout = évite que le modèle “mémorise” trop les données

La sortie :

    1 neurone avec sigmoid → donne une probabilité de toxicité (entre 0 et 1)

Ensuite on configure l’apprentissage :

adam → méthode d’optimisation
binary_crossentropy → adaptée à toxique / non toxique
accuracy → mesure de performance

En résumé : on crée un modèle qui prend un vecteur chimique en entrée et apprend à prédire s’il est toxique ou non.

In [24]:
# 4. CRÉATION DU MODÈLE KERAS
model = keras.Sequential([
    # Couche d'entrée qui correspond à la taille de notre fingerprint (2048)
    keras.Input(shape=(2048,)),
    
    # Première couche cachée avec 64 neurones et activation ReLU
    layers.Dense(64, activation='relu'),
    
    # Couche de régularisation (Dropout) pour éviter le surapprentissage
    layers.Dropout(0.2),
    
    # Deuxième couche cachée avec 32 neurones
    layers.Dense(32, activation='relu'),
    
    # Couche de sortie : 1 neurone avec activation Sigmoid pour une probabilité (0 à 1)
    layers.Dense(1, activation='sigmoid')
])

# Compilation du modèle
# On utilise binary_crossentropy car c'est un problème de classification binaire
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

model.summary()

Model: "sequential_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_9 (Dense)             (None, 64)                131136    
                                                                 
 dropout_3 (Dropout)         (None, 64)                0         
                                                                 
 dense_10 (Dense)            (None, 32)                2080      
                                                                 
 dense_11 (Dense)            (None, 1)                 33        
                                                                 
Total params: 133249 (520.50 KB)
Trainable params: 133249 (520.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


# Explication du nombre de paramètres = (entrées × neurones) + biais
Première couche
dense_6 → (None, 64)    Param # = 131136

- Entrée = 2048 (taille du fingerprint)
- Sortie = 64 neurones

Calcul : (2048 × 64) + 64 = 131072 + 64 = 131136

Interprétation :

2048 poids par neurone
64 neurones → chacun apprend un motif chimique
+64 biais (1 par neurone)

In [25]:
# 5. ENTRAÎNEMENT DU MODÈLE
print("Début de l'entraînement...")
history = model.fit(
    X_train, y_train,
    epochs=50,           # Nombre de passages sur l'ensemble des données
    batch_size=2,        # Mise à jour des poids toutes les 2 molécules
    verbose=1,
    validation_data=(X_test, y_test)
)
print("Entraînement terminé !")

Début de l'entraînement...
Epoch 1/50
2/2 [==============================] - 1s 150ms/step - loss: 0.7032 - accuracy: 0.5000 - val_loss: 0.6869 - val_accuracy: 0.5000
Epoch 2/50
2/2 [==============================] - 0s 33ms/step - loss: 0.6879 - accuracy: 0.7500 - val_loss: 0.6846 - val_accuracy: 0.5000
Epoch 3/50
2/2 [==============================] - 0s 32ms/step - loss: 0.6538 - accuracy: 0.7500 - val_loss: 0.6823 - val_accuracy: 0.5000
Epoch 4/50
2/2 [==============================] - 0s 44ms/step - loss: 0.6416 - accuracy: 1.0000 - val_loss: 0.6790 - val_accuracy: 0.5000
Epoch 5/50
2/2 [==============================] - 0s 41ms/step - loss: 0.6168 - accuracy: 0.7500 - val_loss: 0.6753 - val_accuracy: 0.5000
Epoch 6/50
2/2 [==============================] - 0s 37ms/step - loss: 0.6160 - accuracy: 1.0000 - val_loss: 0.6713 - val_accuracy: 0.5000
Epoch 7/50
2/2 [==============================] - 0s 32ms/step - loss: 0.5833 - accuracy: 1.0000 - val_loss: 0.6671 - val_accuracy: 0.5000

In [26]:
# 6. ÉVALUATION ET PRÉDICTION SUR UNE NOUVELLE MOLÉCULE
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Précision sur le jeu de test : {accuracy * 100:.2f}%\n")

nouvelle_molecule_smiles = "c1ccccc1C" # Toluène
nouvelle_empreinte = smiles_to_fingerprint(nouvelle_molecule_smiles)
nouvelle_empreinte = nouvelle_empreinte.reshape(1, -1) # Keras attend (batch_size, n_features)

# Prédire la probabilité (entre 0 et 1)
probabilite = model.predict(nouvelle_empreinte, verbose=0)[0][0]

print(f"--- Analyse du Toluène ({nouvelle_molecule_smiles}) ---")
if probabilite >= 0.5:
    print(f"Prédiction : TOXIQUE (Confiance : {probabilite*100:.2f}%)")
else:
    print(f"Prédiction : NON TOXIQUE (Confiance : {(1-probabilite)*100:.2f}%)")

Précision sur le jeu de test : 100.00%

--- Analyse du Toluène (c1ccccc1C) ---
Prédiction : TOXIQUE (Confiance : 95.49%)
